# Kafka Word Count — Structured Streaming
Run each cell in order. The producer must already be running.

## 0. Install Kafka connector JAR
Sets the submit args so PySpark pulls the Kafka JAR before the session starts.

In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell"
)
print("PYSPARK_SUBMIT_ARGS set.")

PYSPARK_SUBMIT_ARGS set.


## 1. Create Spark session

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("KafkaWordCount")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
spark

:: loading settings :: url = jar:file:/Users/shishir/anaconda3/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/shishir/.ivy2/cache
The jars for the packages stored in: /Users/shishir/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-529336f8-2c05-4711-8ce3-fb55b5d8d5c8;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 220ms :: artifacts dl 8ms


Spark version: 3.5.1


## 2. Read raw stream from Kafka
Each row Spark receives from Kafka has these columns:

| Column | Type | Description |
|---|---|---|
| key | binary | message key (unused here) |
| **value** | binary | the actual message body |
| topic | string | topic name |
| partition | int | Kafka partition |
| offset | long | position in partition |
| **timestamp** | timestamp | when Kafka received the message |

We print the schema so you can see this structure.

In [3]:
KAFKA_BROKER = "localhost:9092"
TOPIC = "word-count-input"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)

print("Raw Kafka stream schema:")
raw_stream.printSchema()

Raw Kafka stream schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



26/05/02 20:11:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 3. Cast `value` bytes → sentence string
We only keep the columns we need.

In [4]:
from pyspark.sql.functions import col

sentences = raw_stream.select(
    col("value").cast("string").alias("sentence"),
    col("timestamp"),
)

print("After casting value to string:")
sentences.printSchema()

After casting value to string:
root
 |-- sentence: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



## 4. Split sentences → individual words
`explode` turns one row with a list into one row **per element** in that list.

In [5]:
from pyspark.sql.functions import explode, split

words = sentences.select(
    explode(split(col("sentence"), r"\s+")).alias("word"),
    col("timestamp"),
)

print("After explode — one row per word:")
words.printSchema()

After explode — one row per word:
root
 |-- word: string (nullable = false)
 |-- timestamp: timestamp (nullable = true)



## 5. Windowed word count

- **Watermark (20 s)** — Spark waits up to 20 s for late-arriving events before closing a window.
- **Window (30 s, slide 10 s)** — counts words in a 30-second bucket, recalculated every 10 s.
- Each word can appear in **multiple overlapping windows**.

In [6]:
from pyspark.sql.functions import window

word_counts = (
    words
    .withWatermark("timestamp", "20 seconds")
    .groupBy(
        window(col("timestamp"), "30 seconds", "10 seconds"),
        col("word"),
    )
    .count()
    .orderBy("window", "count", ascending=[True, False])
)

print("Word count query plan (logical):")
word_counts.explain()

Word count query plan (logical):
== Physical Plan ==
*(6) Sort [window#33-T20000ms ASC NULLS FIRST, count#32L DESC NULLS LAST], true, 0
+- Exchange rangepartitioning(window#33-T20000ms ASC NULLS FIRST, count#32L DESC NULLS LAST, 2), ENSURE_REQUIREMENTS, [plan_id=98]
   +- *(5) HashAggregate(keys=[window#33-T20000ms, word#25], functions=[count(1)])
      +- StateStoreSave [window#33-T20000ms, word#25], state info [ checkpoint = <unknown>, runId = b14bc4fd-095d-4893-9164-64734f991942, opId = 0, ver = 0, numPartitions = 2], Append, -9223372036854775808, -9223372036854775808, 2
         +- *(4) HashAggregate(keys=[window#33-T20000ms, word#25], functions=[merge_count(1)])
            +- StateStoreRestore [window#33-T20000ms, word#25], state info [ checkpoint = <unknown>, runId = b14bc4fd-095d-4893-9164-64734f991942, opId = 0, ver = 0, numPartitions = 2], 2
               +- *(3) HashAggregate(keys=[window#33-T20000ms, word#25], functions=[merge_count(1)])
                  +- Exchange hashp

## 6. Start the streaming query
Writes each micro-batch to the console every 10 seconds.

> **Keep this cell running.** Interrupt the kernel to stop it.

In [ ]:
query = (
    word_counts.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .option("numRows", 50)
    .trigger(processingTime="10 seconds")
    .start()
)

print(f"Query started. Status: {query.status}")
query.awaitTermination()

26/05/02 20:12:45 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/tr/k980qy9n2kl4d_dmml8c0k780000gn/T/temporary-a7b2add6-88a1-4cc0-b0c1-a0cf9d72b6ae. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/02 20:12:45 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query started. Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


26/05/02 20:12:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+----+-----+
|window|word|count|
+------+----+-----+
+------+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |word      |count|
+------------------------------------------+----------+-----+
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|the       |2    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|mat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|accessible|1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|makes     |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|cat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|pyspark   |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|python    |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|spark     |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|sat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|from      |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|on        |1    |
|{2026-05-02 20:13:10, 2026-05-02 2

...

-------------------------------------------
Batch: 10
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |word      |count|
+------------------------------------------+----------+-----+
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|the       |2    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|accessible|1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|mat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|spark     |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|python    |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|cat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|on        |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|sat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|from      |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|pyspark   |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|makes     |1    |
|{2026-05-02 20:13:10, 2026-05-02 

-------------------------------------------
Batch: 13
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |word      |count|
+------------------------------------------+----------+-----+
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|the       |2    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|accessible|1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|mat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|spark     |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|python    |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|cat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|on        |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|sat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|from      |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|pyspark   |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|makes     |1    |
|{2026-05-02 20:13:10, 2026-05-02 


[Stage 75:>                                                         (0 + 1) / 1]



-------------------------------------------
Batch: 15
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |word      |count|
+------------------------------------------+----------+-----+
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|the       |2    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|accessible|1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|mat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|spark     |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|python    |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|cat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|on        |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|sat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|from      |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|pyspark   |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|makes     |1    |
|{2026-05-02 20:13:10, 2026-05-02 

-------------------------------------------
Batch: 18
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |word      |count|
+------------------------------------------+----------+-----+
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|the       |2    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|accessible|1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|mat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|spark     |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|python    |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|cat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|on        |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|sat       |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|from      |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|pyspark   |1    |
|{2026-05-02 20:13:00, 2026-05-02 20:13:30}|makes     |1    |
|{2026-05-02 20:13:10, 2026-05-02 

## 7. (Optional) Check query status / stop
Run this cell in a **new cell** while the query is running, or after interrupting it.

In [ ]:
# Uncomment to inspect or stop
print(query.status)
print(query.lastProgress)
query.stop()